# NextGen NBA 3.1 — HCP Universe Selection Analysis

**Date:** October 9, 2026  
**Status:** Exploratory Analysis — Transitioning from Product-Based to Indication-Based Universe

---

## 1. NBA 3.0 Baseline (Product-Based Approach)

The HCP universe for NBA 3.0 was defined using a **product-based** methodology.  
**Time frame adopted:** 2 years (May '23 to Apr '25)

| # | Criterion | Description | HCP Count | % of Universe |
|---|-----------|-------------|----------:|:-------------:|
| 1 | **SGLT2/GLP1 Prescribers (Decile 2+)** | All HCPs (decile 2 and above) with SGLT2/GLP1 prescriptions | 184,480 | 83.2% |
| 2 | **Kerendia Writers** | All HCPs prescribing Kerendia in complete history | 49,849 | 22.5% |
| 3 | **NEPH HCPs** | All Nephrologists | 15,209 | 6.9% |
| 4 | **Targeting & BPT HCPs** | All HCPs in BPT or Targeting List | 116,944 | 52.7% |
| | **Total HCP Universe** | Union of all criteria | **221,723** | — |

> *Percentages represent each group's share of the total universe (221,723)*

## 2. NBA 3.1 Current Run (Product-Based, Updated Data)

| Criterion | NBA 3.0 | NBA 3.1 | Delta | Notes |
|-----------|--------:|--------:|------:|-------|
| SGLT2/GLP1 (all) | 648,778 | 575,742 | -73,036 | Pool shrank but Decile 2+ grew |
| SGLT2/GLP1 (Decile 2+) | 184,480 | 193,974 | +9,494 | Higher-volume writers grew |
| Kerendia Writers | 49,849 | 77,972 | +28,123 | \~57% growth in Kerendia adoption |
| NEPH HCPs | 15,209 | 16,414 | +1,205 | Stable |
| Targeting & BPT | 116,944 | 78,389 | -38,555 | Call Plan / BPT list updated |
| **Total Universe** | **221,723** | **232,965** | **+11,242** | **+5.1% growth** |

In [0]:
from pyspark.sql import functions as F
import pandas as pd

CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
SCHEMA = "pace_ker_in_house_schema"

# 2-year lookback window
LOOKBACK_START = "2024-10-01"
LOOKBACK_END = "2026-09-30"

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Lookback: {LOOKBACK_START} to {LOOKBACK_END}")

In [0]:
# === Load all 4 tables ===
lrx_nbrx = spark.table(f"`{CATALOG}`.`{SCHEMA}`.lrx_nbrx")
pt_lvl_nbrx = spark.table(f"`{CATALOG}`.`{SCHEMA}`.pt_lvl_nbrx")
pt_lvl_trx = spark.table(f"`{CATALOG}`.`{SCHEMA}`.pt_lvl_trx")
indication_table = spark.table(f"`{CATALOG}`.`{SCHEMA}`.indication_table")

# === Summary stats ===
summary = []
for name, df, date_col in [
    ("lrx_nbrx", lrx_nbrx, "WEEK_ENDING_DATE"),
    ("pt_lvl_nbrx", pt_lvl_nbrx, "NBRX_DT"),
    ("pt_lvl_trx", pt_lvl_trx, "SVC_DT"),
    ("indication_table", indication_table, None),
]:
    row_count = df.count()
    hcp_count = df.select("PROVIDER_ID").distinct().count() if "PROVIDER_ID" in df.columns else None
    pat_count = df.select("PATIENT_ID").distinct().count() if "PATIENT_ID" in df.columns else None
    if date_col:
        date_range = df.selectExpr(f"MIN({date_col})", f"MAX({date_col})").collect()[0]
        min_dt, max_dt = str(date_range[0]), str(date_range[1])
    else:
        min_dt, max_dt = "N/A", "N/A"
    summary.append({
        "Table": name, "Rows": f"{row_count:,}", "Patients": f"{pat_count:,}" if pat_count else "—",
        "HCPs": f"{hcp_count:,}" if hcp_count else "—", "Min Date": min_dt, "Max Date": max_dt
    })

display(pd.DataFrame(summary))

In [0]:
# === INDICATION_GROUP (8 values, no T1D split) ===
print("INDICATION_GROUP (original — no T1D split):")
lrx_nbrx.select("INDICATION_GROUP").distinct().orderBy("INDICATION_GROUP").show(20, truncate=False)

# === INDICATION_GROUP_APP1 (16 values, with T1D split) ===
print("INDICATION_GROUP_APP1 (recommended — includes T1D):")
lrx_nbrx.select("INDICATION_GROUP_APP1").distinct().orderBy("INDICATION_GROUP_APP1").show(20, truncate=False)

In [0]:
# === T1D flag comparison across 3 versions ===
print("T1D Flag: ORIGINAL vs APP1 vs APP2")
indication_table.groupBy("T1D_FLAG_ORIGINAL", "T1D_FLAG_APP1", "T1D_FLAG_APP2").agg(
    F.count("*").alias("patient_count")
).orderBy(F.desc("patient_count")).show()

# === T2D flag comparison ===
print("T2D Flag: ORIGINAL vs APP1 vs APP2")
indication_table.groupBy("T2D_FLAG_ORIGINAL", "T2D_FLAG_APP1", "T2D_FLAG_APP2").agg(
    F.count("*").alias("patient_count")
).orderBy(F.desc("patient_count")).show()

In [0]:
# === ORIGINAL → APP1 disagreements ===
disagree_app1 = indication_table.filter(F.col("INDICATION_ORIGINAL") != F.col("INDICATION_APP1"))
print(f"Rows where ORIGINAL != APP1: {disagree_app1.count():,}")
print("\nTop ORIGINAL → APP1 reclassifications:")
display(
    disagree_app1.groupBy("INDICATION_ORIGINAL", "INDICATION_APP1").agg(
        F.count("*").alias("patients")
    ).orderBy(F.desc("patients"))
)

In [0]:
# === APP1 → APP2 disagreements ===
disagree_app2 = indication_table.filter(F.col("INDICATION_APP1") != F.col("INDICATION_APP2"))
print(f"Rows where APP1 != APP2: {disagree_app2.count():,}")
print("\nTop APP1 → APP2 relaxations:")
display(
    disagree_app2.groupBy("INDICATION_APP1", "INDICATION_APP2").agg(
        F.count("*").alias("patients")
    ).orderBy(F.desc("patients"))
)

In [0]:
# === Patient counts across all 3 flag versions ===
total_patients = indication_table.count()
print(f"Total patients in indication_table: {total_patients:,}\n")

prevalence = indication_table.selectExpr(
    f"{total_patients} as total_patients",
    # ORIGINAL
    "SUM(CAST(T1D_FLAG_ORIGINAL AS INT)) as T1D_ORIGINAL",
    "SUM(CAST(T2D_FLAG_ORIGINAL AS INT)) as T2D_ORIGINAL",
    "SUM(CAST(CKD_FLAG AS INT)) as CKD_ALL",
    "SUM(CAST(HF_FLAG AS INT)) as HF_ALL",
    # APP1
    "SUM(CAST(T1D_FLAG_APP1 AS INT)) as T1D_APP1",
    "SUM(CAST(T2D_FLAG_APP1 AS INT)) as T2D_APP1",
    # APP2
    "SUM(CAST(T1D_FLAG_APP2 AS INT)) as T1D_APP2",
    "SUM(CAST(T2D_FLAG_APP2 AS INT)) as T2D_APP2"
)
display(prevalence)

---
## 4. Indication-Based EDA Results
**Analysis Window:** 2 years (Oct '24 – Sep '26)  
**Source Table:** `lrx_nbrx` using `INDICATION_GROUP_APP1`

In [0]:
# === Filter lrx_nbrx to 2-year lookback ===
lrx_filtered = lrx_nbrx.filter(
    (F.col("WEEK_ENDING_DATE") >= LOOKBACK_START) & 
    (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END)
)
total_hcps_in_window = lrx_filtered.select("PROVIDER_ID").distinct().count()
print(f"Total HCPs in 2-year window: {total_hcps_in_window:,}")

In [0]:
# === HCPs by INDICATION_GROUP_APP1 ===
hcps_by_indication = lrx_filtered.groupBy("INDICATION_GROUP_APP1").agg(
    F.countDistinct("PROVIDER_ID").alias("n_hcps"),
    F.sum("NBRX").alias("total_nbrx")
).orderBy(F.desc("n_hcps"))

display(hcps_by_indication)

In [0]:
# === HCPs treating each indication (any combination containing it) ===
results = []
for ind_name in ["T1D", "T2D", "CKD", "HF"]:
    cnt = lrx_filtered.filter(
        F.col("INDICATION_GROUP_APP1").contains(ind_name)
    ).select("PROVIDER_ID").distinct().count()
    results.append({"Indication": ind_name, "HCPs": cnt, "% of Total": f"{100*cnt/total_hcps_in_window:.1f}%"})

# Union of all 4
indication_hcps = lrx_filtered.filter(
    F.col("INDICATION_GROUP_APP1") != "OTHERS"
).select("PROVIDER_ID").distinct()
union_count = indication_hcps.count()
results.append({"Indication": "Union (any T1D/T2D/CKD/HF)", "HCPs": union_count, "% of Total": f"{100*union_count/total_hcps_in_window:.1f}%"})

others_only = total_hcps_in_window - union_count
results.append({"Indication": "OTHERS only", "HCPs": others_only, "% of Total": f"{100*others_only/total_hcps_in_window:.1f}%"})

display(pd.DataFrame(results))

In [0]:
# === Cross-indication overlaps ===
hcp_flags = lrx_filtered.select("PROVIDER_ID", "INDICATION_GROUP_APP1").distinct()

t1d_hcps = hcp_flags.filter(F.col("INDICATION_GROUP_APP1").contains("T1D")).select("PROVIDER_ID").distinct()
t2d_hcps = hcp_flags.filter(F.col("INDICATION_GROUP_APP1").contains("T2D")).select("PROVIDER_ID").distinct()
ckd_hcps = hcp_flags.filter(F.col("INDICATION_GROUP_APP1").contains("CKD")).select("PROVIDER_ID").distinct()
hf_hcps = hcp_flags.filter(F.col("INDICATION_GROUP_APP1").contains("HF")).select("PROVIDER_ID").distinct()

overlaps = [
    {"Overlap": "T2D ∩ CKD", "HCP Count": t2d_hcps.intersect(ckd_hcps).count()},
    {"Overlap": "T2D ∩ HF", "HCP Count": t2d_hcps.intersect(hf_hcps).count()},
    {"Overlap": "CKD ∩ HF", "HCP Count": ckd_hcps.intersect(hf_hcps).count()},
    {"Overlap": "T2D ∩ CKD ∩ HF", "HCP Count": t2d_hcps.intersect(ckd_hcps).intersect(hf_hcps).count()},
]
display(pd.DataFrame(overlaps))

In [0]:
# === Top drug classes for each of the 4 target indications ===
for ind_name in ["T2D", "CKD", "HF", "T1D"]:
    print("=" * 70)
    print(f"TOP DRUG CLASSES for {ind_name} (any combination containing {ind_name})")
    print("=" * 70)
    lrx_filtered.filter(
        F.col("INDICATION_GROUP_APP1").contains(ind_name)
    ).groupBy("PRODUCT_CLASS").agg(
        F.countDistinct("PROVIDER_ID").alias("n_hcps"),
        F.sum("NBRX").alias("total_nbrx")
    ).orderBy(F.desc("n_hcps")).show(13, truncate=False)
    print()

In [0]:
# === Kerendia writers by indication ===
ker_filtered = lrx_filtered.filter(F.col("PRODUCT_CLASS") == "KERENDIA")
ker_total = ker_filtered.select("PROVIDER_ID").distinct().count()
print(f"Total Kerendia NBRx writers in 2-year window: {ker_total:,}\n")

ker_by_ind = ker_filtered.groupBy("INDICATION_GROUP_APP1").agg(
    F.countDistinct("PROVIDER_ID").alias("n_hcps"),
    F.sum("NBRX").alias("total_nbrx")
).orderBy(F.desc("n_hcps"))

display(ker_by_ind)

# === Coverage check ===
ker_hcps = ker_filtered.select("PROVIDER_ID").distinct()
ker_in_indication = ker_hcps.intersect(indication_hcps).count()
print(f"\nKerendia writers in indication set: {ker_in_indication:,} ({100*ker_in_indication/ker_total:.1f}%)")
print(f"Kerendia writers ONLY in OTHERS: {ker_total - ker_in_indication}")

### 4.6 Specialty-Based HCP Analysis
The original notebook only included **Nephrologists (NEPH)**. We now expand to **NEPH, CARD, ENDO, PCP, HOSPITALIST**.

In [0]:
# === All specialty groups and their HCP counts ===
spec_summary = lrx_filtered.groupBy("SPECIALTY_GROUP").agg(
    F.countDistinct("PROVIDER_ID").alias("n_hcps"),
    F.sum("NBRX").alias("total_nbrx")
).orderBy(F.desc("n_hcps"))

display(spec_summary)

In [0]:
# === Indication coverage and Kerendia writers per specialty ===
target_specs = ["NEPH", "CARD", "ENDO", "PCP", "HOSPITALIST", "HOS"]
spec_results = []

for spec in target_specs:
    spec_df = lrx_filtered.filter(F.col("SPECIALTY_GROUP") == spec)
    total = spec_df.select("PROVIDER_ID").distinct().count()
    
    row = {"Specialty": spec, "Total HCPs": total}
    for ind in ["T2D", "CKD", "HF", "T1D"]:
        cnt = spec_df.filter(
            F.col("INDICATION_GROUP_APP1").contains(ind)
        ).select("PROVIDER_ID").distinct().count()
        row[f"% {ind}"] = f"{100*cnt/total:.1f}%" if total > 0 else "0%"
    
    ker = spec_df.filter(F.col("PRODUCT_CLASS") == "KERENDIA").select("PROVIDER_ID").distinct().count()
    row["Kerendia Writers"] = ker
    row["Kerendia %"] = f"{100*ker/total:.1f}%" if total > 0 else "0%"
    spec_results.append(row)

display(pd.DataFrame(spec_results))

In [0]:
# === How many specialty HCPs are already in the indication set? ===
overlap_results = []
for spec in ["NEPH", "CARD", "ENDO", "PCP", "HOSPITALIST"]:
    spec_hcps = lrx_filtered.filter(
        F.col("SPECIALTY_GROUP") == spec
    ).select("PROVIDER_ID").distinct()
    total = spec_hcps.count()
    already = spec_hcps.intersect(indication_hcps).count()
    overlap_results.append({
        "Specialty": spec, "Total HCPs": f"{total:,}",
        "Already in Indication Set": f"{already:,}",
        "Net New Adds": f"{total - already:,}",
        "% Already Captured": f"{100*already/total:.1f}%"
    })

# Combined
all_spec = lrx_filtered.filter(
    F.col("SPECIALTY_GROUP").isin(["NEPH", "CARD", "ENDO", "PCP", "HOSPITALIST"])
).select("PROVIDER_ID").distinct()
total_all = all_spec.count()
already_all = all_spec.intersect(indication_hcps).count()
overlap_results.append({
    "Specialty": "COMBINED", "Total HCPs": f"{total_all:,}",
    "Already in Indication Set": f"{already_all:,}",
    "Net New Adds": f"{total_all - already_all:,}",
    "% Already Captured": f"{100*already_all/total_all:.1f}%"
})

display(pd.DataFrame(overlap_results))

In [0]:
# === Head-to-head: SGLT2/GLP1 (product) vs T1D/T2D/CKD/HF (indication) ===
sglt2_glp1_hcps = lrx_filtered.filter(
    F.col("PRODUCT_CLASS").isin(["SGLT2", "GLP1"])
).select("PROVIDER_ID").distinct()

product_count = sglt2_glp1_hcps.count()
indication_count = indication_hcps.count()
overlap_count = sglt2_glp1_hcps.intersect(indication_hcps).count()
gained = indication_hcps.subtract(sglt2_glp1_hcps).count()
lost = sglt2_glp1_hcps.subtract(indication_hcps).count()

comparison = pd.DataFrame([
    {"Metric": "Total HCPs in scope", "Product-Based (SGLT2/GLP1)": f"{product_count:,}", "Indication-Based (T1D/T2D/CKD/HF)": f"{indication_count:,}"},
    {"Metric": "Overlap", "Product-Based (SGLT2/GLP1)": f"{overlap_count:,}", "Indication-Based (T1D/T2D/CKD/HF)": f"{overlap_count:,}"},
    {"Metric": "Unique to this approach", "Product-Based (SGLT2/GLP1)": f"{lost:,}", "Indication-Based (T1D/T2D/CKD/HF)": f"{gained:,}"},
    {"Metric": "Kerendia writer coverage", "Product-Based (SGLT2/GLP1)": "—", "Indication-Based (T1D/T2D/CKD/HF)": f"{100*ker_in_indication/ker_total:.1f}%"},
])
display(comparison)

print(f"\nThe indication approach captures {100*overlap_count/product_count:.1f}% of old SGLT2/GLP1 HCPs")
print(f"while adding {gained:,} new HCPs from other drug classes (ACE, ARB, BB, Diuretics, sMRA, etc.)")

---
## 6. Proposed NBA 3.1 Updated HCP Universe

### New Selection Criteria

| # | Criterion | Source | Description |
|---|-----------|--------|-------------|
| 1 | **Indication-Based HCPs (Decile 2+)** | `lrx_nbrx` | All HCPs prescribing to T1D/T2D/CKD/HF patients, decile 2+ by total NBRx |
| 2 | **Kerendia Writers** | `lrx_nbrx` | All HCPs prescribing Kerendia (complete history) |
| 3 | **Specialty HCPs** | `lrx_nbrx` | NEPH, CARD, ENDO, PCP, HOSPITALIST — sourced directly from `SPECIALTY_GROUP` |
| 4 | **Targeting & BPT HCPs** | `CP_KER_HCP` | All HCPs in BPT or Targeting List (unchanged) |

### Enhanced Output Schema

| Column | Description |
|:-------|:------------|
| `PROVIDER_ID` | Unique HCP identifier |
| `T1D_FLAG` | 1 if HCP treats T1D patients |
| `T2D_FLAG` | 1 if HCP treats T2D patients |
| `CKD_FLAG` | 1 if HCP treats CKD patients |
| `HF_FLAG` | 1 if HCP treats HF patients |
| `KERENDIA_FLAG` | 1 if HCP prescribes Kerendia |
| `NEPH_FLAG` | 1 if Nephrologist |
| `CARD_FLAG` | 1 if Cardiologist |
| `ENDO_FLAG` | 1 if Endocrinologist |
| `PCP_FLAG` | 1 if Primary Care Physician |
| `HOSPITALIST_FLAG` | 1 if Hospitalist |
| `TARGETING_HCP_FLAG` | 1 if in Targeting list |
| `BPT_HCP_FLAG` | 1 if in BPT list |
| `INDICATION_DECILE` | NBRx-based decile rank (1–10) |
| `PULL_DT` | Data pull date |

---
## 7. Next Steps

- [ ] Confirm lookback window for NBA 3.1 (2 years ending when?)
- [ ] Decide on decile methodology: per-indication deciles or combined NBRx decile?
- [ ] Confirm whether to keep `INDICATION_GROUP_APP1` (with T1D) vs `INDICATION_GROUP` (without T1D)
- [ ] Confirm whether to include `HOS` alongside `HOSPITALIST`
- [ ] Build the updated `01_HCP_Universe_Identification` notebook
- [ ] Validate new universe against Kerendia writer and Targeting list coverage

---
*Generated by Genie Code — October 9, 2026*